In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
from huggingface_hub import login
login()

In [ ]:
from transformers import pipeline

pipe = pipeline("text-generation", model="Qwen/Qwen2.5-3B-Instruct")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

In [ ]:
!pip install sentence-transformers PyPDF2 faiss-cpu transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from PyPDF2 import PdfReader
import faiss
import torch

In [ ]:
embedding_model_name = "BAAI/bge-m3"

embedding_model = SentenceTransformer(embedding_model_name)

In [ ]:
import transformers

print("Transformers:", transformers.__version__)

In [ ]:
#step 1 extract data from the pdf
def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        full_text += page.extract_text() + "\n"
    return full_text

In [ ]:
#step 2 parsing = chunking
def chunk_text(text, chunk_size=500, overlap=50):
    words = text.split()
    chunks = []
    step = chunk_size - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

In [ ]:
#step 3 embedding
def embed_chunks(chunks):
    embeddings = embedding_model.encode(chunks, convert_to_numpy=True)

    if isinstance(embeddings, tuple):
        embeddings = embeddings[0]

    return embeddings

In [ ]:
#step 4 vector database
def create_faiss_index(embeddings):
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

In [ ]:
#step 5 get the user's question and look for best answer
def search_index(query, index, chunks, k=3):
    query_embedding = embedding_model.encode([query], convert_to_numpy=True)

    distances, indices = index.search(query_embedding,  k)

    results = []

    for i in indices[0]:
        if i != -1:
            results.append(chunks[i])

    return results

In [ ]:
def generate_text(question, results, max_new_tokens=200):

    context = "\n\n".join(results)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a document question-answering assistant. "
                "Answer the user's question using only the information "
                "provided in the context. "
                "Do not use outside knowledge. "
                "If the answer cannot be found in the context, "
                "say that the information is not available in the document. "
                "Answer in the same language as the user's question."
            )
        },
        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question:\n{question}"
            )
        }
    ]

    output = pipe(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )

    return output[0]["generated_text"][-1]["content"]

In [ ]:
current_chunks = None
current_index = None
current_filename = None

In [ ]:
!pip install fastapi uvicorn pyngrok accelerate -q

In [ ]:
NGROK_TOKEN = "NGROK_TOKEN"
API_KEY = "API_KEY"

In [ ]:
from fastapi import FastAPI, Request, HTTPException, UploadFile, File
import uvicorn, threading, time, socket
from pyngrok import ngrok, conf


In [ ]:
import fastapi
import uvicorn
import pyngrok
import accelerate
import transformers
import sentence_transformers
import faiss
import PyPDF2

In [ ]:
app = FastAPI(
    title="PDF RAG API",
    description="RAG API for question answering over user-uploaded PDFs",
    version="1.0"
)

In [ ]:
@app.post("/upload")
async def upload_pdf(
    request: Request,
    file: UploadFile = File(...)
):
    global current_chunks
    global current_index
    global current_filename

    try:
        # -----------------------------
        # 1. Check API key
        # -----------------------------
        if request.headers.get("authorization") != f"Bearer {API_KEY}":
            raise HTTPException(
                status_code=401,
                detail="Unauthorized"
            )

        print("Upload request received.")
        print("Filename:", file.filename)
        print("Content type:", file.content_type)

        # -----------------------------
        # 2. Check file type
        # -----------------------------
        if file.content_type != "application/pdf":
            raise HTTPException(
                status_code=400,
                detail="Only PDF files are supported."
            )

        # -----------------------------
        # 3. Read PDF
        # -----------------------------
        pdf_bytes = await file.read()

        import io

        print("PDF size:", len(pdf_bytes), "bytes")

        if not pdf_bytes:
            raise HTTPException(
                status_code=400,
                detail="The uploaded PDF is empty."
            )

        # -----------------------------
        # 4. Extract text
        # -----------------------------
        pdf_file = io.BytesIO(pdf_bytes)

        print("Extracting PDF text...")

        text = extract_text_from_pdf(pdf_file)

        print("Extracted characters:", len(text))

        if not text.strip():
            raise HTTPException(
                status_code=400,
                detail=(
                    "Could not extract text from this PDF. "
                    "The PDF may contain scanned images instead of selectable text."
                )
            )

        # -----------------------------
        # 5. Create chunks
        # -----------------------------
        print("Creating chunks...")

        chunks = chunk_text(
            text,
            chunk_size=500,
            overlap=50
        )

        print("Number of chunks:", len(chunks))

        if not chunks:
            raise HTTPException(
                status_code=400,
                detail="No usable text chunks were created from the PDF."
            )

        # -----------------------------
        # 6. Create embeddings
        # -----------------------------
        print("Creating embeddings...")

        embeddings = embed_chunks(chunks)

        print("Embedding shape:", embeddings.shape)

        # -----------------------------
        # 7. Create FAISS index
        # -----------------------------
        print("Creating FAISS index...")

        index = create_faiss_index(embeddings)

        print("FAISS index created.")

        # -----------------------------
        # 8. Store active document
        # -----------------------------
        current_chunks = chunks
        current_index = index
        current_filename = file.filename

        print("Document successfully processed.")

        return {
            "message": "PDF processed successfully.",
            "filename": current_filename,
            "chunks": len(current_chunks)
        }

    except HTTPException:
        raise

    except Exception as e:
        print("====================================")
        print("UPLOAD ERROR:")
        print(type(e).__name__)
        print(str(e))
        print("====================================")

        raise HTTPException(
            status_code=500,
            detail=f"Upload processing failed: {type(e).__name__}: {str(e)}"
        )

In [ ]:
@app.get("/health")
async def health():

    return {
        "status": "ok",
        "message": "PDF RAG API is running"
    }

In [ ]:
@app.post("/generate")
async def gen(req: Request):

    if req.headers.get("authorization") != f"Bearer {API_KEY}":

        raise HTTPException(  status_code=401, detail="Unauthorized")

    data = await req.json()

    question = data.get("question")

    if not question:

        raise HTTPException( status_code=400, detail="Question is required.")

    if current_index is None or current_chunks is None:

        raise HTTPException( status_code=400,  detail="Please upload a PDF first.")

    results = search_index(  question, current_index,  current_chunks,  k=3)
    
    if not results:

        raise HTTPException( status_code=404,  detail="No relevant information was found in the document.")

    answer = generate_text( question, results)

    return {
        "response": answer,
        "sources": results
    }

In [ ]:
def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
conf.get_default().auth_token = NGROK_TOKEN
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

print("\nHealth endpoint:")
print(public_url + "/health")

print("\nUpload endpoint:")
print(public_url + "/upload")

print("\nGenerate endpoint:")
print(public_url + "/generate")

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(2)